In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark==3.5.1

import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-11-openjdk-amd64"

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("FbOM Cleaning") \
    .getOrCreate()



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.0/317.0 MB 6.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 16.0 MB/s eta 0:00:00
  Created wheel for pyspark: filename=pyspark-3.5.1-py2.py3-none-any.whl size=317488493 sha256=964ef8d326cb6f70b3077ce62a7c6b7b186ed4964a2b0907e4c67005ae944948
  Stored in directory: /root/.cache/pip/wheels/b1/91/5f/283b53010a8016a4ff1c4a1edd99bbe73afacb099645b5471b
Successfully built pyspark
  Attempting uninstall: py4j
    Found existing installation: py4j 0.10.9.9
    Uninstalling py4j-0.10.9.9:
      Successfully uninstalled py4j-0.10.9.9
  Attempting uninstall: pyspark
    Found existing installation: pyspark 4.0.2
    Uninstalling pyspark-4.0.2:
      Successfully uninstalled pyspark-4.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-conn

In [4]:
input_path = "/content/drive/MyDrive/data/part1.jsonl.gz"
df = spark.read.json(input_path)

print("Schema:")
df.printSchema()


initial_count = df.count()
print("Initial reviews:", initial_count)

Schema:
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

Initial reviews: 21943472


In [5]:
from pyspark.sql.functions import *


# 1. Gộp title + text

df = df.withColumn(
    "review_text",
    concat_ws(" ", col("title"), col("text"))
)


# 2. Xóa review rỗng

df = df.filter(col("review_text").isNotNull())
df = df.filter(trim(col("review_text")) != "")


# 3. Xóa duplicate

df = df.dropDuplicates(["review_text"])

after_clean_count = df.count()


# 4. Chuẩn hóa text

df = df.withColumn("review_text", lower(col("review_text")))

df = df.withColumn(
    "review_text",
    regexp_replace(col("review_text"), r"[^a-z0-9\s\.\!\?]", " ")
)

df = df.withColumn(
    "review_text",
    regexp_replace(col("review_text"), r"\s+", " ")
)

df = df.withColumn("review_text", trim(col("review_text")))

In [6]:
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number, monotonically_increasing_id


window_review = Window.orderBy(monotonically_increasing_id())

df = df.withColumn("review_id", row_number().over(window_review))

In [7]:
window_sentence = Window.orderBy(monotonically_increasing_id())

df = df.withColumn("sentence_id", row_number().over(window_sentence))

In [8]:
# Tách câu
df = df.withColumn(
    "sentences",
    split(col("review_text"), r"[\.!\?]+")
)

df = df.withColumn("sentence_text", explode(col("sentences")))

df = df.withColumn("sentence_text", trim(col("sentence_text")))
df = df.filter(col("sentence_text") != "")

In [9]:

df_final = df.select(
    col("parent_asin"),
    col("review_id"),
    col("sentence_id"),
    col("sentence_text"),
    col("rating")
)

sentence_count = df_final.count()

df_final.show(5)

+-----------+---------+-----------+--------------------+------+
|parent_asin|review_id|sentence_id|       sentence_text|rating|
+-----------+---------+-----------+--------------------+------+
| B01AU72OGU|        1|          1|great background ...|   3.0|
| B01AU72OGU|        1|          1|i also will steam it|   3.0|
| B01AU72OGU|        1|          1|however it came d...|   3.0|
| B01AU72OGU|        1|          1|i m pretty bummed...|   3.0|
| B01AU72OGU|        1|          1|edit the mark is ...|   3.0|
+-----------+---------+-----------+--------------------+------+
only showing top 5 rows



In [10]:
output_parquet = "/content/drive/MyDrive/outputs/cleaned_part1.parquet"

df_final.write.mode("overwrite").parquet(output_parquet)



✅ Saved parquet


In [14]:
import os
import gzip


def get_folder_size(path):
    total = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            total += os.path.getsize(os.path.join(dirpath, f))
    return total

def get_uncompressed_size_gzip(file_path):
    total_size = 0
    with gzip.open(file_path, 'rb') as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b''):
            total_size += len(chunk)
    return total_size

input_size = get_uncompressed_size_gzip(input_path) / (1024**3)
output_size = get_folder_size(output_parquet) / (1024**3)

report_path = "/content/drive/MyDrive/outputs/reports/reviews_cleaning1_report_Electronics.txt"

os.makedirs("/content/drive/MyDrive/outputs/reports", exist_ok=True)

with open(report_path, "w", encoding="utf-8") as f:
    f.write(f"Số review ban đầu: {initial_count:,}\n")
    f.write(f"Số review còn lại: {after_clean_count:,}\n")
    f.write(f"Số câu sau khi tách: {sentence_count:,}\n")
    f.write(f"Kích thước ban đầu: {input_size:.2f}GB\n")
    f.write(f"Kích thước cuối cùng: {output_size:.2f}GB\n")


In [16]:
!git config --global user.email "thuhungminh@gmail.com"
!git config --global user.name "hoangvuu1"

In [17]:
!git clone https://github.com/TranTheHung2312332/FbOM-from-amazon-ds-PTIT.git

Cloning into 'FbOM-from-amazon-ds-PTIT'...
remote: Enumerating objects: 132, done.
remote: Counting objects: 100% (44/44), done.
remote: Compressing objects: 100% (40/40), done.
remote: Total 132 (delta 10), reused 23 (delta 3), pack-reused 88 (from 1)
Receiving objects: 100% (132/132), 30.21 MiB | 15.74 MiB/s, done.
Resolving deltas: 100% (45/45), done.


In [18]:
%cd FbOM-from-amazon-ds-PTIT

/content/FbOM-from-amazon-ds-PTIT


In [19]:
!git checkout -b electronics_1

Switched to a new branch 'electronics_1'


In [34]:
!cp !cp "/content/drive/MyDrive/Colab Notebooks/Electronics_category1.ipynb" .
!cp /content/drive/MyDrive/outputs/reports/reviews_cleaning1_report_Electronics.txt .


cp: cannot stat '!cp': No such file or directory


In [29]:
for root, dirs, files in os.walk("/content"):
    for f in files:
        if ".ipynb" in f:
            print(os.path.join(root, f))

/content/drive/MyDrive/THDeeplearning1.ipynb
/content/drive/MyDrive/THDeeplearning1 (1) (1).ipynb
/content/drive/MyDrive/THDeeplearning1 (1).ipynb
/content/drive/MyDrive/Colab Notebooks/Electronics_category1.ipynb
/content/drive/MyDrive/Colab Notebooks/ptkpTH1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled0.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled2.ipynb
/content/drive/MyDrive/Colab Notebooks/THDeeplearning1.ipynb
/content/drive/MyDrive/Colab Notebooks/Untitled3.ipynb
/content/FbOM-from-amazon-ds-PTIT/notebooks/data_exploration_4.ipynb
/content/FbOM-from-amazon-ds-PTIT/notebooks/data_exploration_2.ipynb
/content/FbOM-from-amazon-ds-PTIT/notebooks/data_exploration_1.ipynb
/content/FbOM-from-amazon-ds-PTIT/notebooks/data_exploration_3.ipynb
